#📌 Extracão

In [ ]:
import pandas as pd

# URL da API (arquivo JSON)
url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science/main/TelecomX_Data.json"

# Carregar dados
dados = pd.read_json(url)

# Visualizar primeiras linhas
dados.head()

# Conhecendo o dataset

In [ ]:
dados.info()

In [ ]:
dados.columns

# Verificando inconsistências

In [ ]:
dados.isnull().sum()

In [ ]:
dados["customerID"].duplicated().sum()

In [ ]:
dados["Churn"].unique()

Como algumas colunas do dataset estavam estruturadas em formato de dicionário, foi necessário realizar a normalização dessas informações para facilitar a manipulação dos dados.

# 🔧 Transformação

Normalizando os dicionários

In [ ]:
df = pd.json_normalize(dados.to_dict(orient="records"))

# Tratando inconsistências

Corrigir Churn

In [ ]:
# Substituindo strings vazias e removendo nulos
df["Churn"] = df["Churn"].replace("", pd.NA)
df = df.dropna(subset=["Churn"])
df["Churn"].unique()

Padronizando churn

In [ ]:
df["Churn"] = df["Churn"].str.lower()

Convertendo em valores numéricos

In [ ]:
df["account.Charges.Monthly"] = pd.to_numeric(df["account.Charges.Monthly"], errors="coerce")

df["account.Charges.Total"] = pd.to_numeric(df["account.Charges.Total"], errors="coerce")

Verificando nulos

In [ ]:
df.isnull().sum()

Para ampliar a análise dos dados financeiros dos clientes, foi criada a coluna Contas_Diarias. Essa variável foi calculada a partir do valor de cobrança mensal dividido por 30 dias, permitindo observar uma estimativa do custo diário dos serviços contratados.

In [ ]:
df["Contas_Diarias"] = df["account.Charges.Monthly"] / 30

df[["account.Charges.Monthly", "Contas_Diarias"]].head()

In [ ]:
df["Contas_Diarias"].describe()

Nesta etapa foi realizada uma padronização de algumas variáveis do conjunto de dados. A variável Churn foi convertida para um formato binário, facilitando futuras análises quantitativas. Além disso, algumas categorias foram traduzidas e determinadas colunas foram renomeadas para tornar o conjunto de dados mais claro e compreensível durante a análise exploratória.

Converter Churn para binário

In [ ]:
df["Churn_binario"] = df["Churn"].map({"yes": 1, "no": 0})

df[["Churn", "Churn_binario"]].head()

Traduzindo algumas categorias

In [ ]:
df["account.Contract"] = df["account.Contract"].replace({
    "Month-to-month": "Mensal",
    "One year": "1 ano",
    "Two year": "2 anos"
})

Renomeando algumas colunas

In [ ]:
df = df.rename(columns={
    "customer.tenure": "Tempo_Cliente",
    "account.Charges.Monthly": "Mensalidade",
    "account.Charges.Total": "Total_Pago"
})

In [ ]:
df.head()

#📊 Carga e análise

Nesta etapa foi realizada uma análise descritiva do conjunto de dados com o objetivo de compreender melhor a distribuição das variáveis numéricas e categóricas presentes no dataset. Foram utilizadas métricas estatísticas como média, desvio padrão, valores mínimos e máximos, além da frequência das categorias, permitindo uma visão inicial sobre o comportamento dos clientes e suas características.

Análise das colunas numéricas

In [ ]:
df.describe()

Análise das colunas categóricas

In [ ]:
df.describe(include="object")

Churn no dataset

In [ ]:
df["Churn"].value_counts()

Calculando a distribuição

In [ ]:
churn_dist = df["Churn"].value_counts()

churn_dist

In [ ]:
df["Churn"].value_counts(normalize=True) * 100

Para compreender melhor o comportamento de evasão dos clientes, foi analisada a distribuição da variável Churn, que indica se o cliente permaneceu ou cancelou o serviço. A visualização permite identificar a proporção entre clientes que continuam utilizando os serviços da empresa e aqueles que optaram pelo cancelamento.

Gráfico

In [ ]:
import matplotlib.pyplot as plt

churn_dist.plot(kind="bar")

plt.title("Distribuição de Evasão de Clientes")
plt.xlabel("Churn")
plt.ylabel("Quantidade de Clientes")

plt.show()

Observa-se que a maior parte dos clientes permanece na empresa, enquanto uma parcela menor representa os casos de evasão. Essa informação é importante para direcionar análises posteriores sobre os fatores que podem influenciar o cancelamento dos serviços.

# Contagem de Evasão por Variáveis Categóricas

Agora, vamos explorar como a evasão se distribui de acordo com variáveis categóricas, como gênero, tipo de contrato, método de pagamento, entre outras. Essa análise pode revelar padrões interessantes, como, por exemplo, se clientes de determinados perfis têm maior tendência a cancelar o serviço, o que ajudará a direcionar ações estratégicas.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Configurando o estilo dos gráficos
sns.set_theme(style="whitegrid")

# Lista das principais variáveis categóricas para analisar
colunas_categoricas = ['customer.gender', 'account.Contract', 'account.PaymentMethod', 'internet.InternetService']

# Criando uma figura com subplots (2 linhas, 2 colunas)
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 12))
axes = axes.flatten()

# Loop para gerar um gráfico para cada variável categórica
for i, col in enumerate(colunas_categoricas):
    sns.countplot(data=df, x=col, hue='Churn', ax=axes[i], palette=["#1f77b4", "#ff7f0e"])

    axes[i].set_title(f'Evasão por {col}', fontsize=14)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Quantidade de Clientes')

    # Rotacionar os rótulos do eixo X se for a coluna de método de pagamento (para não sobrepor os textos)
    if col == 'account.PaymentMethod':
        axes[i].tick_params(axis='x', rotation=45)

# Ajustar o layout para não cortar os textos
plt.tight_layout()
plt.show()

Para complementar a visualização, podemos observar os números exatos do cruzamento entre o Tipo de Contrato e o Churn:

In [ ]:
# Tabela cruzada: Contrato vs Churn
tabela_contrato = pd.crosstab(df['account.Contract'], df['Churn'], margins=True, margins_name="Total")
display(tabela_contrato)

# Tabela cruzada em percentual (linha)
tabela_contrato_pct = pd.crosstab(df['account.Contract'], df['Churn'], normalize='index') * 100
display(tabela_contrato_pct.round(2))

### 💡 Primeiros Insights da Análise Categórica
* **Gênero:** A proporção de cancelamento parece ser muito semelhante entre homens e mulheres, indicando que o gênero não é um fator determinante para o Churn.
* **Tipo de Contrato:** Clientes com contrato **Mensal** apresentam uma taxa de evasão significativamente maior do que aqueles com contratos anuais ou de 2 anos.
* **Método de Pagamento:** Clientes que utilizam *Electronic check* (cheque eletrônico) concentram o maior volume de cancelamentos.

# 🌟 Extra: Análise de Correlação entre Variáveis

Como passo adicional, vamos explorar a correlação matemática entre as variáveis numéricas do dataset e a nossa variável alvo (`Churn_binario`).

Para enriquecer essa análise, primeiro vamos criar uma nova variável chamada **`Total_Servicos`**, que soma quantos serviços adicionais o cliente possui (telefone, múltiplas linhas, segurança online, backup, suporte técnico, streaming de TV e filmes).

In [ ]:
# Lista com as colunas referentes aos serviços oferecidos
colunas_servicos = [
    'phone.PhoneService', 'phone.MultipleLines',
    'internet.OnlineSecurity', 'internet.OnlineBackup',
    'internet.DeviceProtection', 'internet.TechSupport',
    'internet.StreamingTV', 'internet.StreamingMovies'
]

# Criando a coluna de Total de Serviços
df['Total_Servicos'] = df[colunas_servicos].apply(lambda x: x.map({'Yes': 1})).fillna(0).sum(axis=1)

# GATILHO DE SEGURANÇA: Garantindo que o Churn_binario existe antes de chamar
if 'Churn_binario' not in df.columns:
    # Tenta mapear considerando que o texto pode estar minúsculo ou com a inicial maiúscula
    df["Churn_binario"] = df["Churn"].str.lower().map({"yes": 1, "no": 0})

# Verificando as novas colunas
df[['customerID', 'Total_Servicos', 'Churn_binario']].head()

In [ ]:
# Selecionando apenas as colunas numéricas de interesse
colunas_numericas = ['Tempo_Cliente', 'Mensalidade', 'Total_Pago',
                     'Contas_Diarias', 'Total_Servicos', 'Churn_binario']

# Calculando a correlação de Pearson
matriz_correlacao = df[colunas_numericas].corr()

# Configurando e gerando o Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(matriz_correlacao, annot=True, cmap='RdBu_r', vmin=-1, vmax=1, fmt=".2f", linewidths=0.5)

plt.title('Matriz de Correlação (Heatmap)', fontsize=16)
plt.show()

### Aprofundando: Serviços x Churn
Para visualizar de forma mais clara como a quantidade de serviços afeta a evasão, vamos gerar um gráfico que mostra a taxa média de Churn de acordo com o total de serviços contratados.

In [ ]:
plt.figure(figsize=(10, 5))
# barplot mostra por padrão a média do eixo Y (neste caso, a taxa de evasão de 0 a 1)
sns.barplot(data=df, x='Total_Servicos', y='Churn_binario', palette='viridis', errorbar=None)

plt.title('Taxa de Evasão por Quantidade de Serviços Contratados', fontsize=14)
plt.xlabel('Total de Serviços Contratados')
plt.ylabel('Taxa Média de Churn (Evasão)')

plt.show()

----------------------------------------------------------------------------------------------------------------------------------------------------------------

# 📄 Relatório Final: Análise de Evasão de Clientes (Churn) - TelecomX

## 1. Introdução
A retenção de clientes é um dos maiores desafios estratégicos para empresas de telecomunicações. O objetivo deste projeto foi analisar a base de dados de clientes da TelecomX para entender o comportamento de evasão (**Churn**). Ao identificar os perfis de clientes com maior probabilidade de cancelar os serviços e as variáveis que mais influenciam essa decisão, a empresa pode adotar medidas proativas para melhorar a satisfação, reduzir perdas financeiras e direcionar campanhas de retenção de forma mais eficiente.

## 2. Limpeza e Tratamento de Dados
Para garantir a integridade da análise, os dados brutos passaram por um processo de preparação rigoroso:
* **Extração e Normalização:** Os dados foram consumidos de uma API (formato JSON) e normalizados, achatando colunas estruturadas em dicionários para o formato tabular.
* **Tratamento de Inconsistências:** Identificamos e removemos registros com valores em branco na variável alvo (`Churn`), garantindo que a análise não fosse enviesada.
* **Engenharia de Recursos (Feature Engineering):** * Criamos a variável `Contas_Diarias` para entender o impacto financeiro diário por cliente.
    * Criamos a variável `Total_Servicos`, somando todos os serviços adicionais contratados, para medir o nível de engajamento do cliente com o portfólio da empresa.
* **Transformações de Tipo e Padronização:** Conversão de dados financeiros para o tipo numérico, padronização do `Churn` para formato binário (0 e 1) e tradução de variáveis categóricas para o português.

## 3. Análise Exploratória de Dados (EDA)
A exploração dos dados foi dividida em duas frentes:
* **Análise Categórica:** Avaliamos a distribuição do Churn cruzando com variáveis como Gênero, Tipo de Contrato, Método de Pagamento e Tipo de Internet, utilizando gráficos de barras.
* **Análise de Correlação:** Utilizamos uma Matriz de Correlação (Heatmap) para medir a força da relação matemática entre variáveis quantitativas (Tempo de Cliente, Mensalidade, Total de Serviços) e a probabilidade de evasão.

## 4. Conclusões e Principais Insights
A análise revelou padrões de comportamento muito claros:
1. **O Tipo de Contrato dita o ritmo do Churn:** Clientes com contratos na modalidade **"Mensal"** representam a esmagadora maioria das evasões. Contratos anuais ou de dois anos garantem uma retenção significativamente superior.
2. **Fidelização por Combos (Lock-in):** A matriz de correlação e os gráficos de serviços evidenciaram que clientes com apenas 1 serviço contratado possuem a maior probabilidade de evasão. À medida que mais serviços são adicionados ao pacote (telefone, internet, segurança, streaming), a taxa de Churn cai drasticamente.
3. **Alerta no Método de Pagamento:** Usuários que pagam via **Electronic check** (cheque eletrônico) cancelam o serviço em uma proporção muito maior em comparação aos métodos automáticos (cartão de crédito ou débito em conta).
4. **Tempo de Casa (Tenure):** Existe uma correlação negativa moderada (-0.35) entre o tempo do cliente e o churn. Isso confirma que os clientes mais novos são o maior grupo de risco.
5. **Serviço de Internet:** Clientes que utilizam Fibra Óptica (*Fiber optic*) apresentam uma taxa de cancelamento superior aos usuários de internet via cabo (DSL).
6. **Demografia:** O Gênero não é um fator decisivo, pois as taxas de evasão são praticamente idênticas entre homens e mulheres.

## 5. Recomendações Estratégicas
Com base nos dados levantados, sugerimos as seguintes ações para a TelecomX:
* **Campanhas de Upsell e Combos:** Criar ofertas agressivas incentivando clientes que possuem apenas um serviço a adicionarem pacotes extras (como streaming ou segurança) por um custo marginal reduzido. O objetivo é aumentar a "amarração" (lock-in) do cliente com a empresa.
* **Incentivo a Contratos Longos:** Oferecer benefícios claros (descontos ou upgrades de velocidade gratuitos) para clientes que aceitarem migrar do contrato "Mensal" para o de "1 ano", focando principalmente nos clientes novos (primeiros meses).
* **Revisão da Jornada de Pagamento:** Investigar imediatamente a experiência de pagamento via *Electronic check*. Pode haver atritos ou falhas de usabilidade frustrando o cliente. Incentivar o débito automático pode ajudar a reter esse público.
* **Auditoria Técnica na Fibra Óptica:** A equipe técnica deve investigar se há instabilidade na rede, problemas de instalação ou se a relação custo-benefício percebida da fibra óptica está abaixo da expectativa em comparação com a concorrência.